In [1]:
%run ./imports.ipynb
%run ./tools_fun.ipynb
%run ./tools_handle.ipynb
%run ./db_config.ipynb
%run ./config.ipynb

In [3]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

In [4]:
def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    
    response = gemini.chat.completions.create(
        model=model,
        messages=messages,
        tools=tools
    )

    if response.choices[0].finish_reason == "tool_calls":
        assistant_message = response.choices[0].message
        
        # 2. Get the list of tool responses
        tool_responses = handle_tool_calls(assistant_message)
        
        messages.append(assistant_message)
        messages.extend(tool_responses) 
        
        response = gemini.chat.completions.create(
            model=model, 
            messages=messages
        )
    
    final_content = response.choices[0].message.content
    
    return final_content if final_content is not None else "Operation completed."

In [5]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


DATABASE TOOL CALLED: Getting price for London
